# Training Model Klasifikasi Dokumen ALMEX\nNotebook untuk melatih model `jenis_pipeline` dan `arah_pipeline`.\n\n## Struktur Dataset\n```\ndataset/\n├── jenis/\n│   ├── surat_masuk/      (*.txt hasil OCR)\n│   ├── surat_keluar/\n│   ├── invoice/\n│   └── surat_jalan/\n└── arah/\n    ├── surat_masuk/\n    └── surat_keluar/\n```

In [ ]:
import os\nimport glob\nimport random\nimport numpy as np\nimport pandas as pd\nimport joblib\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import LinearSVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score\n\nRANDOM_STATE = 42\nnp.random.seed(RANDOM_STATE)\nrandom.seed(RANDOM_STATE)

## 1. Load Dataset

In [ ]:
def load_text_dataset(base_path):\n    """Load semua .txt dari subfolder sebagai (text, label)."""\n    texts, labels = [], []\n    for label in sorted(os.listdir(base_path)):\n        label_path = os.path.join(base_path, label)\n        if not os.path.isdir(label_path):\n            continue\n        files = glob.glob(os.path.join(label_path, '*.txt'))\n        for f in files:\n            try:\n                with open(f, 'r', encoding='utf-8') as fh:\n                    texts.append(fh.read())\n                    labels.append(label)\n            except Exception as e:\n                print(f'Error baca {f}: {e}')\n    return texts, labels\n\n# Load jenis dan arah\njenis_texts, jenis_labels = load_text_dataset('./dataset/jenis')\narah_texts, arah_labels = load_text_dataset('./dataset/arah')\n\nprint('Jenis:', len(jenis_texts), 'samples')\nprint(pd.Series(jenis_labels).value_counts())\nprint('\nArah:', len(arah_texts), 'samples')\nprint(pd.Series(arah_labels).value_counts())

## 2. Preprocessing & Train/Test Split

In [ ]:
def make_splits(texts, labels, test_size=0.2):\n    return train_test_split(\n        texts, labels,\n        test_size=test_size,\n        stratify=labels,\n        random_state=RANDOM_STATE\n    )\n\nXj_train, Xj_test, yj_train, yj_test = make_splits(jenis_texts, jenis_labels)\nXa_train, Xa_test, ya_train, ya_test = make_splits(arah_texts, arah_labels)\n\nprint(f'Jenis  -> train: {len(Xj_train)}, test: {len(Xj_test)}')\nprint(f'Arah   -> train: {len(Xa_train)}, test: {len(Xa_test)}')

## 3. Build Pipeline & Training\nGunakan TF-IDF + LinearSVC (cepat, akurat untuk teks).

In [ ]:
def build_pipeline(C=1.0):\n    return Pipeline([\n        ('tfidf', TfidfVectorizer(\n            lowercase=True,\n            ngram_range=(1, 2),\n            max_df=0.95,\n            min_df=2,\n            sublinear_tf=True\n        )),\n        ('clf', LinearSVC(C=C, random_state=RANDOM_STATE))\n    ])\n\njenis_pipeline = build_pipeline(C=1.0)\narah_pipeline = build_pipeline(C=1.0)\n\n# Training\njenis_pipeline.fit(Xj_train, yj_train)\narah_pipeline.fit(Xa_train, ya_train)\n\nprint('Training selesai.')

## 4. Evaluasi Dasar (Accuracy, F1, Classification Report)

In [ ]:
def evaluate(pipeline, X_test, y_test, title='Model'):\n    y_pred = pipeline.predict(X_test)\n    acc = accuracy_score(y_test, y_pred)\n    f1 = f1_score(y_test, y_pred, average='weighted')\n    print(f'=== {title} ===')\n    print(f'Accuracy : {acc:.4f}')\n    print(f'F1 Score : {f1:.4f}\n')\n    print(classification_report(y_test, y_pred, zero_division=0))\n    return y_pred\n\nyj_pred = evaluate(jenis_pipeline, Xj_test, yj_test, 'Jenis Dokumen')\nya_pred = evaluate(arah_pipeline, Xa_test, ya_test, 'Arah Dokumen')

## 5. Confusion Matrix

In [ ]:
def plot_confusion(y_true, y_pred, labels, title):\n    cm = confusion_matrix(y_true, y_pred, labels=labels)\n    plt.figure(figsize=(6, 5))\n    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',\n                xticklabels=labels, yticklabels=labels)\n    plt.title(f'Confusion Matrix - {title}')\n    plt.ylabel('Actual')\n    plt.xlabel('Predicted')\n    plt.tight_layout()\n    plt.show()\n\nplot_confusion(yj_test, yj_pred, sorted(set(yj_test)), 'Jenis Dokumen')\nplot_confusion(ya_test, ya_pred, sorted(set(ya_test)), 'Arah Dokumen')

## 6. Cross Validation (5-Fold)

In [ ]:
def run_cv(pipeline, texts, labels, cv=5):\n    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)\n    scores = cross_val_score(pipeline, texts, labels, cv=skf, scoring='f1_weighted')\n    print(f'CV F1 (weighted): {scores.mean():.4f} (+/- {scores.std():.4f})')\n    return scores\n\nprint('Jenis:')\nrun_cv(build_pipeline(), jenis_texts, jenis_labels)\nprint('\nArah:')\nrun_cv(build_pipeline(), arah_texts, arah_labels)

## 7. Robustness Testing\nSimulasi kondisi nyata: keyword dihapus, header dihilangkan, error OCR.

In [ ]:
import re\n\ndef delete_keywords(text, keywords=['invoice', 'surat jalan', 'nomor', 'kepada']):\n    """Hapus keyword penting untuk lihat dependensi model."""\n    t = text\n    for kw in keywords:\n        t = re.sub(re.escape(kw), '', t, flags=re.IGNORECASE)\n    return t\n\ndef remove_header(text, lines=3):\n    """Hilangkan N baris pertama (kop surat/header)."""\n    return '\n'.join(text.splitlines()[lines:])\n\ndef simulate_ocr_error(text, prob=0.1):\n    """Simulasi typo OCR: ganti karakter acak."""\n    chars = list(text)\n    for i in range(len(chars)):\n        if random.random() < prob:\n            if chars[i].isalpha():\n                chars[i] = random.choice('abcdefghijklmnopqrstuvwxyz')\n    return ''.join(chars)\n\ndef test_robustness(pipeline, X, y, name, transform_fn):\n    X_mod = [transform_fn(t) for t in X]\n    y_pred = pipeline.predict(X_mod)\n    acc = accuracy_score(y, y_pred)\n    print(f'{name}: Accuracy = {acc:.4f}')\n    return acc\n\nprint('=== ROBUSTNESS: JENIS ===')\ntest_robustness(jenis_pipeline, Xj_test, yj_test, 'Keyword deleted', delete_keywords)\ntest_robustness(jenis_pipeline, Xj_test, yj_test, 'Header removed', remove_header)\ntest_robustness(jenis_pipeline, Xj_test, yj_test, 'OCR error 10%', simulate_ocr_error)\n\nprint('\n=== ROBUSTNESS: ARAH ===')\ntest_robustness(arah_pipeline, Xa_test, ya_test, 'Keyword deleted', delete_keywords)\ntest_robustness(arah_pipeline, Xa_test, ya_test, 'Header removed', remove_header)\ntest_robustness(arah_pipeline, Xa_test, ya_test, 'OCR error 10%', simulate_ocr_error)

## 8. Save Model ke Backend

In [ ]:
os.makedirs('../backend/ml_model', exist_ok=True)\n\njoblib.dump(jenis_pipeline, '../backend/ml_model/jenis_pipeline.pkl')\njoblib.dump(arah_pipeline, '../backend/ml_model/arah_pipeline.pkl')\n\nprint('Model tersimpan di ../backend/ml_model/')\nprint('Files:', os.listdir('../backend/ml_model'))

---\n**Catatan:** Kalau dataset masih berupa gambar (PDF/JPG), extract dulu ke `.txt` pakai OCR (Tesseract/EasyOCR) sebelum training.